# Process weather and demand data

## Import packages

In [ ]:
import numpy as np
import pyarrow.parquet as pq
import joblib
import scipy
import pandas as pd
from pandas import DatetimeIndex
import time
import datetime
import os
import re
import glob
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pytz
import yaml
import pprint
from collections import defaultdict
import psutil 
import resource  

import os, time, gc
import joblib

from src import input_ops
from src import physics_ops 
from src import file_ops

## Define functions

In [ ]:
  
def export_dict_to_joblib(data_dict, filename="exported_dict.joblib"):
    """
    Saves a dictionary to a .joblib file in the current working directory.
    
    Parameters:
    - data_dict (dict): The dictionary to save.
    - filename (str): Name of the output file (default: 'exported_dict.joblib')
    """
    cwd = os.getcwd()
    output_path = os.path.join(cwd, filename)
    joblib.dump(data_dict, output_path)
    print(f"Dictionary saved to: {output_path}")
    
def expand_TGW_years_scenarios(config):
    """
    Return TGW_years_scenarios as:
        {
            "1990": ["historical"],
            ...
            "2059": ["rcp45hotter"]
        }

    Supports either:
      1. TGW_years_scenarios_ranges
      2. existing TGW_years_scenarios
    """
    if "TGW_years_scenarios_ranges" in config:
        TGW_years_scenarios = {}

        for range_spec in config["TGW_years_scenarios_ranges"]:
            start_year = int(range_spec["start_year"])
            end_year = int(range_spec["end_year"])
            scenarios = range_spec["scenarios"]

            for year in range(start_year, end_year + 1):
                TGW_years_scenarios[str(year)] = scenarios

        return TGW_years_scenarios

    elif "TGW_years_scenarios" in config:
        return config["TGW_years_scenarios"]

    else:
        raise KeyError(
            "Config must contain either 'TGW_years_scenarios_ranges' "
            "or 'TGW_years_scenarios'."
        )

def build_regional_demand_weather_filename(TGW_years_scenarios):
    """
    Build dynamic output filename based on TGW_years_scenarios.

    Example:
        {
            "1990": ["historical"],
            ...
            "2019": ["historical"],
            "2030": ["rcp45hotter"],
            ...
            "2059": ["rcp45hotter"]
        }

    becomes:
        regional_demand_weather_all_cities_1990_2019_historical_2030_2059_rcp45hotter.joblib
    """

    # Invert year -> scenarios into scenario -> years
    scenario_to_years = {}

    for year, scenarios in TGW_years_scenarios.items():
        year_int = int(year)

        for scenario in scenarios:
            if scenario not in scenario_to_years:
                scenario_to_years[scenario] = []

            scenario_to_years[scenario].append(year_int)

    filename_parts = ["regional_demand_weather_all_cities"]

    # Sort by earliest year for stable filename ordering
    for scenario, years in sorted(
        scenario_to_years.items(),
        key=lambda item: min(item[1])
    ):
        years = sorted(years)

        # Detect contiguous year ranges
        start_year = years[0]
        previous_year = years[0]

        for current_year in years[1:] + [None]:
            if current_year is not None and current_year == previous_year + 1:
                previous_year = current_year
            else:
                # Close current contiguous range
                end_year = previous_year

                if start_year == end_year:
                    filename_parts.append(f"{start_year}_{scenario}")
                else:
                    filename_parts.append(f"{start_year}_{end_year}_{scenario}")

                # Start next range if there is one
                if current_year is not None:
                    start_year = current_year
                    previous_year = current_year

    filename = "_".join(filename_parts) + ".joblib"

    return filename

## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

aggregation_level = config['aggregation_level']

demand_mode = config['demand_mode']

CITY_REGIONS_TO_RUN = config['CITY_REGIONS_TO_RUN']

TGW_years_scenarios = expand_TGW_years_scenarios(config)

regional_demand_weather_filename = build_regional_demand_weather_filename(TGW_years_scenarios)

smart_ds_years = config['smart_ds_years']
smart_ds_year = config['smart_ds_years'][0]

smart_ds_load_path = config['smart_ds_load_path'] + f"/{smart_ds_years[0]}" # path to procesed smart-ds resstock data 

input_data_prediction_path = config['input_data_prediction_path']

## Aggregate predicted feeder cooling at city level 

In [ ]:
start_time = time.time()

# Set paths to cooling feeder predicted results
output_path_prediction_str ='main_folder/load_prediction/results/data/prediction/output/2018/months_1_12/cooling_kw_sum/X_columns_D/feeder/'
prediction_model_str = 'MLP_2_512_64'

### --- Initialize dictionaries ---
cooling_loaded_predictions_dict = {}
aggregated_city_feeder_cooling_ts = {}

## --- Load predicted feeder cooling data ---
for TGW_weather_year, TGW_scenarios in TGW_years_scenarios.items():
    for TGW_scenario in TGW_scenarios:
        ## Load cooling data
        predictions_dir = os.path.join(output_path_prediction_str, f"{TGW_scenario}/predictions")
        cooling_loaded_predictions_dict[(TGW_weather_year,TGW_scenario)] = joblib.load(os.path.join(predictions_dir, f"{prediction_model_str}_TGW_{TGW_weather_year}_models_dict.joblib"))

## --- remove regions not in CITY_REGIONS_TO_RUN --- 
def filter_cooling_loaded_predictions(cooling_loaded_predictions_dict, city_regions_to_run):
    """
    Filters cooling_loaded_predictions_dict to keep only entries whose (city, region) 
    appear in city_regions_to_run.
    
    Modifies the dictionary in place.
    """
    for outer_key, inner_dict in list(cooling_loaded_predictions_dict.items()):
        filtered_inner = {}
        for key, df in inner_dict.items():
            # key format: (smartds_year, city, region, feeder, bldg_type)
            _, city, region, _, _ = key
            if city in city_regions_to_run and region in city_regions_to_run[city]:
                filtered_inner[key] = df
        cooling_loaded_predictions_dict[outer_key] = filtered_inner
    
    return cooling_loaded_predictions_dict

cooling_loaded_predictions_dict = filter_cooling_loaded_predictions(cooling_loaded_predictions_dict,CITY_REGIONS_TO_RUN)

## --- Aggregate data at city level ---        
# Loop over each climate scenario
for climate_key, records in cooling_loaded_predictions_dict.items():
    tgw_year, tgw_scenario = climate_key
    city_grouped = defaultdict(list)  # stores tuples of (cooling, temperature) series per city
    aggregated_city_feeder_cooling_ts[(tgw_year, tgw_scenario)] = {}
    # Loop over each feeder-level record
    for inner_key, df in records.items():
        smartds_year, city, region, feeder, bldg_type = inner_key
        df = df.copy()
        if not pd.api.types.is_datetime64_any_dtype(df['date_time']):
            df['date_time'] = pd.to_datetime(df['date_time'])
        df.set_index('date_time', inplace=True)
        # Extract relevant columns
        cooling_series = df['cooling_kw_sum_predicted']
        temp_series = df['Dry Bulb Temperature [°C]']
        city_grouped[city].append((cooling_series, temp_series))
    # For each city, aggregate across feeders/buildings
    for city, ts_pairs in city_grouped.items():
        cooling_ts_list = [pair[0] for pair in ts_pairs]
        temp_ts_list = [pair[1] for pair in ts_pairs]
        # Sum predicted cooling demand, mean temperature
        combined_cooling = pd.concat(cooling_ts_list, axis=1).sum(axis=1)
        combined_temp = pd.concat(temp_ts_list, axis=1).mean(axis=1)
        df_out = pd.DataFrame({
            'cooling_kw_sum_predicted': combined_cooling,
            'Dry Bulb Temperature [°C]': combined_temp
        })
        aggregated_city_feeder_cooling_ts[(tgw_year, tgw_scenario)][city] = df_out
        
del cooling_loaded_predictions_dict
end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")

## Aggregate buildings predicted data at city level 

In [ ]:
ACCUM_DTYPE = np.float32          
COLS = ['cooling_kw_sum_predicted','heating_kw_predicted','total_kw_predicted']

start_time = time.time()

feeder_predictions_path = (
    f"main_folder/load_prediction/results/data/prediction/output/"
    f"{config['smart_ds_years'][0]}/months_{config['start_month']}_{config['end_month']}/"
    f"cooling_n_heating/{config['X_columns_set']}/{config['aggregation_level']}/"
)

# Dictionary to store aggregated buildings cooling, heating and total demand
aggregated_city_predicted_cool_n_heat_dict = {}

for TGW_weather_year, TGW_scenarios in TGW_years_scenarios.items():
    for TGW_scenario in TGW_scenarios:
        print(f"\n=== TGW_year: {TGW_weather_year} | scenario: {TGW_scenario} ===")
        # Initialize internal dictionary for each climate scenario
        aggregated_city_predicted_cool_n_heat_dict[(TGW_weather_year, TGW_scenario)] = {}

        for city, regions in CITY_REGIONS_TO_RUN.items():
            print(f"\nAggregating city: {city}")
            city_index = None
            city_accum = None   
            building_count = 0

            # Stream regions
            for region in regions:
                # Set path to current climate-region buildings data
                predictions_dir = os.path.join(feeder_predictions_path, f"{TGW_scenario}/predictions/{city}/{region}/")
                region_path = os.path.join(predictions_dir, f"{demand_mode}_TGW_{TGW_weather_year}_buildings_cool_n_heat_dict.joblib")

                ## Load one region dict, aggregate immediately, then free data
                region_dict = joblib.load(region_path)
                
                for feeder_key, bldg_dict in region_dict.items():
                    _, _city_chk, _region_chk, feeder_id, _btype = feeder_key # feeder_key = (year, city, region, feeder_id, building_type)

                    # Build a fast lookup: building_id -> count
                    feeder_path_name = input_ops.add_feeder_upper_folder(feeder_id)
                    feeder_path = (
                        f"main_folder/SMART-DS/v1.0/{smart_ds_year}/"
                        f"{city}/{region}/scenarios/base_timeseries/opendss/{feeder_path_name}"
                    )
                    df_cnt = input_ops.count_building_id_occurrences([feeder_path])
                    count_map = df_cnt.set_index('building_id')['count']  # Series for O(1) lookup
                    
                    # Iterate buildings
                    for bldg_id, df in bldg_dict.items():
                        cnt = count_map.get(bldg_id)
                        if pd.isna(cnt):
                            continue

                        # Establish the city's index and accumulator once (from the first seen frame)
                        if city_index is None:
                            city_index = df.index
                            # Pre-allocate accumulator: T x 3 (for COLS)
                            city_accum = np.zeros((len(city_index), len(COLS)), dtype=ACCUM_DTYPE)
                        else:
                            #  shape/index check
                            if len(df.index) != len(city_index) or not df.index.equals(city_index):
                                raise ValueError(f"Time index mismatch in {city}-{region}-{feeder_id}-{bldg_id}")

                        # Pull values, multiply by count, and accumulate (NumPy = no DataFrame allocation)
                        vals = df[COLS].to_numpy(dtype=ACCUM_DTYPE, copy=False)
                        # Scalar multiply, then accumulate
                        city_accum += vals * ACCUM_DTYPE(cnt)

                        building_count += 1

                    # Free per-feeder small objects promptly
                    del df_cnt, count_map
                    gc.collect()

                # Free the entire region dict promptly
                del region_dict
                gc.collect()

            print(f"--- {city}: aggregated {building_count} buildings ---")

            # Materialize final DataFrame only once per city
            if city_accum is None:
                # No data found; store empty frame
                city_df = pd.DataFrame(columns=COLS)
            else:
                city_df = pd.DataFrame(city_accum, index=city_index, columns=COLS)

            aggregated_city_predicted_cool_n_heat_dict[(TGW_weather_year, TGW_scenario)][city] = city_df

city = "all_cities"
aggregated_city_predicted_cool_n_heat_dict_path = smart_ds_load_path + f"/{city}/aggregated_demand/aggregated_city_predicted_cool_n_heat_dict_{city}"
export_dict_to_joblib(aggregated_city_predicted_cool_n_heat_dict, aggregated_city_predicted_cool_n_heat_dict_path)

end_time = time.time()
print("Runtime:", (end_time - start_time) / 60, "minutes")

## Combine weather and aggregated predicted feeder and buildings demand

In [ ]:
start_time = time.time()

# Expected final columns
EXPECTED_COLUMNS = [
    'year',
    'month',
    'day',
    'hour',
    'weekday',
    'weekend',
    'Relative Humidity [%]',
    'Dry Bulb Temperature [°C]',
    'Global Horizontal Radiation [W/m2]',
    'Wind Speed [m/s]',
    'aggregated_predicted_feeder_cooling_kw_sum',
    'aggregated_predicted_buildings_cooling_kw_sum',
    'aggregated_predicted_buildings_heating_kw',
    'aggregated_predicted_buildings_total_kw',
]

WEATHER_COLUMNS = [
    'year',
    'month',
    'day',
    'hour',
    'weekday',
    'weekend',
    'Relative Humidity [%]',
    'Dry Bulb Temperature [°C]',
    'Global Horizontal Radiation [W/m2]',
    'Wind Speed [m/s]',
]

# Initialize dictionary
regional_demand_weather = {}

for TGW_weather_year, TGW_scenarios in TGW_years_scenarios.items():
    for TGW_scenario in TGW_scenarios:
        regional_demand_weather[(TGW_weather_year, TGW_scenario)] = {}

        for city, regions in CITY_REGIONS_TO_RUN.items():

            print(f"---- Loading data for {TGW_weather_year} {TGW_scenario} {city} ---")

            # ------------------------------------------------------------------
            # Add TGW weather data
            # ------------------------------------------------------------------
            TGW_location = {
                "GSO": "Greensboro",
                "AUS": "Austin",
                "SFO": "SanFrancisco",
            }.get(city, city)

            TGW_weather_df_save_path = f"{input_data_prediction_path}/{TGW_location}/{TGW_scenario}/"
            TGW_weather_df = joblib.load(
                os.path.join(
                    TGW_weather_df_save_path,
                    f"TGW_weather_{TGW_weather_year}.joblib",
                )
            )

            TGW_weather_df['date_time'] = pd.to_datetime(TGW_weather_df['date_time'])
            TGW_weather_df = TGW_weather_df.set_index('date_time')

            TGW_weather_df = TGW_weather_df[
                ~((TGW_weather_df.index.month == 2) & (TGW_weather_df.index.day == 29))
            ].copy()

            df_city = TGW_weather_df[WEATHER_COLUMNS].copy()

            # ------------------------------------------------------------------
            # Add aggregated predicted feeder cooling demand
            # ------------------------------------------------------------------
            pred_feeder_cooling_df = aggregated_city_feeder_cooling_ts[
                (TGW_weather_year, TGW_scenario)
            ][city].copy()

            if not isinstance(pred_feeder_cooling_df.index, pd.DatetimeIndex):
                pred_feeder_cooling_df.index = pd.to_datetime(pred_feeder_cooling_df.index)

            pred_feeder_cooling_df = pred_feeder_cooling_df[
                ~(
                    (pred_feeder_cooling_df.index.month == 2)
                    & (pred_feeder_cooling_df.index.day == 29)
                )
            ].copy()

            # Direct index alignment should work because this is the same TGW year/scenario
            df_city.loc[:, 'aggregated_predicted_feeder_cooling_kw_sum'] = (
                pred_feeder_cooling_df['cooling_kw_sum_predicted']
            )

            # ------------------------------------------------------------------
            # Add aggregated predicted buildings cooling, heating, and total demand
            # ------------------------------------------------------------------
            predicted_df = aggregated_city_predicted_cool_n_heat_dict[
                (TGW_weather_year, TGW_scenario)
            ][city][
                [
                    'cooling_kw_sum_predicted',
                    'heating_kw_predicted',
                    'total_kw_predicted',
                ]
            ].copy()

            if not isinstance(predicted_df.index, pd.DatetimeIndex):
                predicted_df.index = pd.to_datetime(predicted_df.index)

            predicted_df = predicted_df[
                ~((predicted_df.index.month == 2) & (predicted_df.index.day == 29))
            ].copy()

            # Align by month-day-hour because predicted building-load timestamps may
            # use a different base year than the TGW weather dataframe
            df_city['mdh'] = list(
                zip(df_city.index.month, df_city.index.day, df_city.index.hour)
            )
            predicted_df['mdh'] = list(
                zip(predicted_df.index.month, predicted_df.index.day, predicted_df.index.hour)
            )

            # Keep one row per month-day-hour
            predicted_df_unique = predicted_df.drop_duplicates(subset='mdh')

            df_city = df_city.merge(
                predicted_df_unique.set_index('mdh'),
                on='mdh',
                how='left',
            )

            df_city = df_city.drop(columns='mdh')

            df_city = df_city.rename(
                columns={
                    'cooling_kw_sum_predicted': 'aggregated_predicted_buildings_cooling_kw_sum',
                    'heating_kw_predicted': 'aggregated_predicted_buildings_heating_kw',
                    'total_kw_predicted': 'aggregated_predicted_buildings_total_kw',
                }
            )

            # Keep only expected final columns, in fixed order
            df_city = df_city[EXPECTED_COLUMNS]

            regional_demand_weather[(TGW_weather_year, TGW_scenario)][city] = df_city

# ----------------------------------------------------------------------
# Export dictionary to joblib
# ----------------------------------------------------------------------
city = "all_cities"
regional_demand_weather_path = (
    smart_ds_load_path
    + f"/{city}/aggregated_demand/"
    + regional_demand_weather_filename
)

export_dict_to_joblib(regional_demand_weather, regional_demand_weather_path)

end_time = time.time()
print("Runtime:", (end_time - start_time) / 60, "minutes")